# HERMES on Google Colab

This notebook runs `train_hermes.py` on a Colab GPU and stores `data/`, `checkpoints/`, and `runs/` on Google Drive so runs can resume across runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/golf-colab'
SOURCE_REPO_ON_DRIVE = f'{DRIVE_ROOT}/repo'
REPO_URL = ''  # Optional: set this if the repo is on GitHub.
REPO_DIR = '/content/golf'
RUN_NAME = 'hermes_v1_fineweb_ode8'

print('DRIVE_ROOT =', DRIVE_ROOT)
print('SOURCE_REPO_ON_DRIVE =', SOURCE_REPO_ON_DRIVE)
print('REPO_DIR   =', REPO_DIR)
print('RUN_NAME   =', RUN_NAME)

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q datasets huggingface_hub pyarrow

In [ ]:
import os
import pathlib
import shutil
import subprocess

for path in [DRIVE_ROOT, SOURCE_REPO_ON_DRIVE, f'{DRIVE_ROOT}/data', f'{DRIVE_ROOT}/checkpoints', f'{DRIVE_ROOT}/runs']:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

if os.path.isdir(os.path.join(SOURCE_REPO_ON_DRIVE, '.git')) or os.path.exists(os.path.join(SOURCE_REPO_ON_DRIVE, 'train_hermes.py')):
    shutil.copytree(
        SOURCE_REPO_ON_DRIVE,
        REPO_DIR,
        ignore=shutil.ignore_patterns('data', 'checkpoints', 'runs', '__pycache__', '.ipynb_checkpoints')
    )
elif REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    raise RuntimeError(
        'Provide REPO_URL or copy this repo into MyDrive/golf-colab/repo before running setup.'
    )

for name in ['data', 'checkpoints', 'runs']:
    repo_path = os.path.join(REPO_DIR, name)
    drive_path = os.path.join(DRIVE_ROOT, name)
    if os.path.islink(repo_path):
        continue
    if os.path.isdir(repo_path):
        shutil.rmtree(repo_path)
    elif os.path.exists(repo_path):
        os.remove(repo_path)
    os.symlink(drive_path, repo_path)

print('Repo ready at', REPO_DIR)
print('Drive-backed data at', DRIVE_ROOT)

In [ ]:
%cd /content/golf
!python3 - <<'PY'
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    if hasattr(torch.cuda, 'is_bf16_supported'):
        print('bf16 supported:', torch.cuda.is_bf16_supported())
PY

## Optional: prepare FineWeb text slices in Colab

Skip this cell if `data/fineweb_train` and `data/fineweb_val` already exist on Drive.

In [ ]:
%cd /content/golf
!python3 prepare_fineweb.py --subset CC-MAIN-2024-10 --format text --train-mb 64 --val-mb 8

## Train or resume

This mirrors your local command and resumes from the latest checkpoint inside `checkpoints/<run_name>/`.

In [ ]:
%cd /content/golf
!python3 train_hermes.py \
  --run_name "$RUN_NAME" \
  --train_path data/fineweb_train \
  --val_path data/fineweb_val \
  --batch 96 \
  --steps 10000 \
  --ode_steps 8 \
  --checkpoint_dir checkpoints \
  --results_file runs/hermes_runs.jsonl \
  --resume latest

In [ ]:
%cd /content/golf
!find checkpoints -maxdepth 2 -type f | sort | tail -n 10
!tail -n 5 runs/hermes_runs.jsonl